# DA_PROJECT: Day 2 – Building the Data Cleaning Module

Today I worked on creating the `data_cleaning.py` module. The goal was to build a reusable, robust data cleaning pipeline to prepare raw data for analysis.

I referred to resources like *“Python Data Wrangling with Pandas”* and online blogs about data-cleaning best practices.

##  Why Data Cleaning?

Raw datasets often come with issues: duplicates, missing values, wrong data types, outliers, and inconsistent columns. Without cleaning, analysis and modeling results may be misleading.

So a structured cleaning pipeline ensures data quality before exploration or modeling.

In [ ]:
import pandas as pd
import numpy as np
from helpers import safe_execute
from data_loader import df_copy

class DataCleaning:

    @safe_execute
    def __init__(self):
        self.df = df_copy()

    @safe_execute
    def remove_duplicates(self):
        self.df = self.df.drop_duplicates()
        return self.df

    @safe_execute
    def handle_missing_values(self, strategy="drop", custom_value=None):
        if strategy == "drop":
            self.df = self.df.dropna()

        elif strategy == "fill_mean":
            mean_vals = self.df.mean(numeric_only=True)
            self.df = self.df.fillna(mean_vals)

        elif strategy == "fill_median":
            median_vals = self.df.median(numeric_only=True)
            self.df = self.df.fillna(median_vals)

        elif strategy == "fill_mode":
            mode_vals = self.df.mode().iloc[0]
            self.df = self.df.fillna(mode_vals)

        elif strategy == "fill_custom":
            if custom_value is None:
                raise ValueError("Pass custom_value for fill_custom strategy")
            self.df = self.df.fillna(custom_value)

        else:
            raise ValueError("Invalid strategy")

        return self.df

    @safe_execute
    def drop_columns(self, col_list):
        if type(col_list) not in [list, tuple]:
            raise ValueError("col_list must be a list")
        self.df = self.df.drop(columns=col_list, errors="ignore")
        return self.df

    @safe_execute
    def rename_columns(self, rename_dict):
        if not isinstance(rename_dict, dict):
            raise ValueError("rename_dict must be a dictionary")
        self.df = self.df.rename(columns=rename_dict)
        return self.df

    @safe_execute
    def change_datatypes(self, dtype_dict):
        if not isinstance(dtype_dict, dict):
            raise ValueError("dtype_dict must be a dictionary")
        for col, dtype in dtype_dict.items():
            self.df[col] = self.df[col].astype(dtype)
        return self.df

    @safe_execute
    def handle_outliers(self, columns=None):
        if columns is None:
            columns = self.df.select_dtypes(include=np.number).columns.tolist()

        for col in columns:
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.df = self.df[(self.df[col] >= lower) & (self.df[col] <= upper)]

        return self.df

    @safe_execute
    def reset_index(self):
        self.df = self.df.reset_index(drop=True)
        return self.df

    @safe_execute
    def get_cleaned_data(self):
        return self.df.copy()

    @safe_execute
    def preview_changes(self, n=5):
        return self.df.head(n)

##  What we built today
- Removed duplicates (if any).
- Implemented flexible missing-value strategies: drop, fill mean/median/mode or custom value.
- Ability to drop unnecessary columns, rename columns, and change data types — all in one pipeline.
- Built simple outlier removal using IQR method across numeric columns.
- Reset Index, preview cleaned data, and retrieve cleaned DataFrame safely.


##  What I Learned & Reflected
- Clean code structure using OOP and decorators improves reusability.
- Handling missing values and outliers carefully — essential for reliable analysis.
- Importance of error handling — `safe_execute` helps prevent crashes during data operations.
- Data cleaning is a crucial step before any EDA or modeling — reinforces data quality discipline.


##  Reference & Further Reading
- Article on data cleaning best practices: *Practical Data Cleaning for Data Scientists* — some concepts inspired today’s design.  
- Pandas documentation for handling missing data & outliers.
